In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [ ]:
# state definition for the batsman workflow
class BatsmanState(TypedDict):
    name: str
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    bpb: float  # balls per boundary
    boundary_percentage: float
    summary: str

In [ ]:
def calculate_strike_rate(state: BatsmanState) -> BatsmanState:
    sr = (state['runs'] / state['balls']) * 100
    return {'strike_rate': sr}

In [ ]:
def calculate_bpb(state: BatsmanState) -> BatsmanState:
    total_boundaries = state['fours'] + state['sixes']
    if total_boundaries > 0:
        bpb = state['balls'] / total_boundaries
    else:
        bpb = float('inf')  # No boundaries hit
    return {'bpb': bpb}

In [ ]:
def calculate_boundary_percentage(state: BatsmanState) -> BatsmanState:
    total_boundaries = state['fours'] + state['sixes']
    if state['balls'] > 0:
        boundary_percentage = (total_boundaries / state['balls']) * 100
    else:
        boundary_percentage = 0.0  # No balls faced
    return {'boundary_percentage': boundary_percentage} 

In [ ]:
def summarize_performance(state: BatsmanState) -> BatsmanState:
    summary = (f"{state['name']} scored {state['runs']} runs off {state['balls']} balls, "
               f"with a strike rate of {state['strike_rate']:.2f}, "
               f"balls per boundary of {state['bpb']:.2f}, "
               f"and a boundary percentage of {state['boundary_percentage']:.2f}%.")
    return {'summary': summary}

In [ ]:
graph = StateGraph(BatsmanState)

# Add nodes to the graph
graph.add_node('calculate_strike_rate', calculate_strike_rate)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percentage', calculate_boundary_percentage)
graph.add_node('summarize_performance', summarize_performance)

# add edges to define the workflow
graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percentage') 

graph.add_edge('calculate_strike_rate', 'summarize_performance')
graph.add_edge('calculate_bpb', 'summarize_performance')
graph.add_edge('calculate_boundary_percentage', 'summarize_performance')

graph.add_edge('summarize_performance', END)

workflow = graph.compile()  # visualize the workflow graph


In [ ]:
intial_state = {
    'name': 'Virat Kohli',
    'runs': 75,
    'balls': 50,
    'fours': 8,
    'sixes': 2
}

workflow.invoke(intial_state)  # execute the workflow with the initial state